# Meshroom Reconstruction

To use meshroom dense reconstruction, I downloaded the binaries from the [website](https://github.com/alicevision/meshroom/releases) and ran the reconstruction

I oppened the executable, selected the Photogrametry option, selected the output folder, dropped the images and clicked in run.

The good results are in the textured mesh.

## Visualizing the result

In [1]:
# Convert Meshroom EXR textures to PNG for Open3D
import os
import shutil
from pathlib import Path
import numpy as np

try:
    import imageio.v3 as iio
    print('Using imageio for EXR conversion')
except ImportError:
    raise ImportError('Please install imageio: pip install imageio[all]')

# Path to your Meshroom OBJ file (update as needed)
obj_path = Path('../reconstructions/Meshroom_Segmented/MeshroomCache/Texturing/f397d1bcca84a9348d1c5ff6f4e3656e73e6ceab/texturedMesh.obj')

# Validate core files exist
if not obj_path.exists():
    raise FileNotFoundError(f'OBJ file not found: {obj_path}')

mtl_path = obj_path.with_suffix('.mtl')
if not mtl_path.exists():
    raise FileNotFoundError(f'MTL file not found: {mtl_path}')

def _convert_exr_to_png(exr_path: Path) -> Path:
    """Load EXR texture, tone-map to 8-bit, and write PNG."""
    print(f'Converting {exr_path.name} to PNG...')
    texture = iio.imread(exr_path)
    
    # Handle different data types and convert to 8-bit
    if texture.dtype in (np.float32, np.float64):
        # Tone-map HDR to LDR
        texture = np.clip(texture, 0.0, 1.0)
        texture = (texture * 255.0).astype(np.uint8)
    elif texture.dtype != np.uint8:
        texture = texture.astype(np.uint8)
    
    # Ensure RGB format
    if texture.ndim == 2:
        texture = np.stack([texture]*3, axis=-1)
    elif texture.shape[-1] == 4:
        texture = texture[..., :3]  # Remove alpha channel
    
    png_path = exr_path.with_suffix('.png')
    iio.imwrite(png_path, texture)
    print(f'  → Saved {png_path.name}')
    return png_path

# Update MTL so Open3D can locate PNG textures
with open(mtl_path, 'r', encoding='utf-8') as handle:
    mtl_lines = handle.readlines()

updated = False
for idx, line in enumerate(mtl_lines):
    tokens = line.strip().split(maxsplit=1)
    if tokens and tokens[0].lower() == 'map_kd' and len(tokens) == 2:
        tex_rel_path = tokens[1]
        tex_path = (mtl_path.parent / tex_rel_path).resolve()
        if tex_path.suffix.lower() == '.exr' and tex_path.exists():
            png_path = _convert_exr_to_png(Path(tex_path))
            rel_png = os.path.relpath(png_path, mtl_path.parent).replace('\\', '/')
            mtl_lines[idx] = f'map_Kd {rel_png}\n'
            updated = True

if updated:
    backup_path = mtl_path.with_suffix('.mtl.bak')
    if not backup_path.exists():
        shutil.copy2(mtl_path, backup_path)
    with open(mtl_path, 'w', encoding='utf-8') as handle:
        handle.writelines(mtl_lines)
    print(f'\nConverted EXR textures to PNG and updated MTL.')
    print(f'Original MTL backed up to: {backup_path}')
else:
    print('No EXR textures found to convert.')

Using imageio for EXR conversion
Converting texture_1001.exr to PNG...
  → Saved texture_1001.png

Converted EXR textures to PNG and updated MTL.
Original MTL backed up to: ..\reconstructions\Meshroom_Segmented\MeshroomCache\Texturing\f397d1bcca84a9348d1c5ff6f4e3656e73e6ceab\texturedMesh.mtl.bak
  → Saved texture_1001.png

Converted EXR textures to PNG and updated MTL.
Original MTL backed up to: ..\reconstructions\Meshroom_Segmented\MeshroomCache\Texturing\f397d1bcca84a9348d1c5ff6f4e3656e73e6ceab\texturedMesh.mtl.bak


In [4]:
# Visualize colored OBJ mesh using Open3D
import open3d as o3d
import os

# Path to your Meshroom OBJ file (update as needed)
obj_path = '../reconstructions/meshroom_20_True/texturedMesh.obj'  # <-- Change if your path is different

# Check if file exists
if not os.path.exists(obj_path):
    raise FileNotFoundError(f'OBJ file not found: {obj_path}')

# Load the mesh (Open3D will try to load textures if referenced in the .mtl file)
mesh = o3d.io.read_triangle_mesh(obj_path, enable_post_processing=True)

# Check if mesh has vertex colors or textures
if mesh.has_vertex_colors():
    print('Mesh has vertex colors.')
elif mesh.has_textures():
    print('Mesh has textures.')
else:
    print('Mesh has no vertex colors or textures.')

# Visualize the mesh
o3d.visualization.draw_geometries([mesh], window_name='Meshroom OBJ Visualization')

Mesh has textures.


## Remarks

The reconstruction is absolutely wild: 9/10. The downsides are: heavy software, need of a GPU, lack of easiness to export the colored textured output.